# 02 train

Steps 4-7 - cache activations, train D1-D4, pool-size check, freeze the method.

DEV concepts only. Nothing here may look at TEST.

## Настройка

In [1]:
%load_ext autoreload
%autoreload 2

import glob
import json
import sys

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from steering import denoiser, generate, hooks, interventions, io, judge, metrics, spaces, vectors

MODEL, LAYER = "gpt2", 6
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL).eval().to(DEVICE)

sae = vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                        layer=LAYER, d_model=768, device=DEVICE)

split = vectors.load_split(io.RESULTS / "feature_splits_gpt2.json")
center = spaces.should_center(MODEL)

prompts = json.loads((io.REPO_ROOT / "configs" / "prompts_neutral_32.json").read_text())["prompts"]

with hooks.ResidualHook(model, layer=LAYER, capture=True) as _h, torch.no_grad():
    model(**tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE))
scale = spaces.activation_scale(_h.captured[0], exclude_sink=True)

pq = glob.glob(str(io.REPO_ROOT.parent / "**/pile-10k*/**/*.parquet"), recursive=True) \
    or glob.glob(str(__import__("pathlib").Path.home() /
                      ".cache/huggingface/hub/datasets--NeelNanda--pile-10k/snapshots/*/data/*.parquet"))
corpus_texts = pd.read_parquet(pq[0])["text"].head(3000).tolist()

print(f"python  {sys.version.split()[0]}")
print(f"device  {DEVICE}")
print(f"split   DEV {len(split.dev)}, TEST {len(split.test)}, fingerprint {split.fingerprint()}")
print(f"prompts {len(prompts)}, corpus {len(corpus_texts)} docs, scale {scale:.1f}, center {center}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

python  3.12.0
device  mps
split   DEV 35, TEST 70, fingerprint ce65a9496b253ceb
prompts 32, corpus 3000 docs, scale 95.1, center True


## Шаг 4. Собираем обучающую выборку активаций.

In [2]:
from steering import denoiser

activations = io.run_or_load(
    "clean_activations_gpt2", {"model": "gpt2", "layer": LAYER, "n_docs": len(corpus_texts),
                                "max_length": 64, "max_tokens": 200_000, "version": 1},
    lambda: denoiser.cache_activations(model, tokenizer, layer=LAYER, texts=corpus_texts,
                                       max_tokens=200_000, batch_size=16, max_length=64),
    where="artifacts",
)
print(f"{activations.shape[0]} activations cached, d_model={activations.shape[1]}")
print(f"median ||h|| = {float(activations.norm(dim=-1).median()):.1f}  "
      f"(should be close to scale={scale:.1f} from earlier)")

cached  clean_activations_gpt2.pt  (6be99263c06aee12)
186699 activations cached, d_model=768
median ||h|| = 89.5  (should be close to scale=95.1 from earlier)


## Шаг 5. Обучение денойзера на разных типах шума

In [3]:
from steering import corruptions

N_VAL = 5000
val_activations = activations[:N_VAL]
train_activations = activations[N_VAL:]
print(f"train pool {train_activations.shape[0]}, held-out val {val_activations.shape[0]}")

decoder = sae.W_dec.detach()

families = {
    "D1": corruptions.Gaussian(sigma_min=0.05, sigma_max=3.0),
    "D2": corruptions.VariancePreserving(),
    "D3": corruptions.FixedPoolRank1(decoder, split, pool_size=256, seed=0),
    "D4": corruptions.FullPoolRank1(decoder, split),
}
for name, c in families.items():
    print(f"{name}: {c.describe()}")

train pool 181699, held-out val 5000
D1: {'corruption': 'D1', 'sigma_min': 0.05, 'sigma_max': 3.0}
D2: {'corruption': 'D2'}
D3: {'corruption': 'D3', 'rho_min': 0.05, 'rho_max': 3.0, 'pool_size': 256, 'pool_seed': 0}
D4: {'corruption': 'D4', 'rho_min': 0.05, 'rho_max': 3.0, 'pool_size': 130967}


In [4]:
STEPS, BATCH_SIZE, LR, SEED = 5000, 256, 1e-3, 0
results = {}

for name, corruption in families.items():
    model, history = denoiser.train_denoiser(
        train_activations, corruption, d_model=768, activation_scale=scale, center=center,
        steps=STEPS, batch_size=BATCH_SIZE, lr=LR, seed=SEED, device=DEVICE, log_every=STEPS // 10,
    )
    val_loss = denoiser.evaluate_denoiser(model, val_activations, corruption,
                                          n_examples=4096, seed=999, device=DEVICE)
    results[name] = {"model": model, "history": history, "val_loss": val_loss,
                     "final_train_loss": history[-1]["loss"],
                     "identity_gap": history[-1]["identity_gap"]}
    print(f"{name}: train_loss={history[-1]['loss']:.4f}  val_loss={val_loss:.4f}  "
          f"identity_gap={history[-1]['identity_gap']:.2f}")

D1: train_loss=0.1979  val_loss=0.2022  identity_gap=6.92
D2: train_loss=0.2887  val_loss=0.2927  identity_gap=6.68
D3: train_loss=0.0215  val_loss=0.0200  identity_gap=3.29
D4: train_loss=0.2159  val_loss=0.1988  identity_gap=6.82


In [5]:
import pandas as pd

rows = []
for name, r in results.items():
    denoiser.save_denoiser(
        r["model"], io.ARTIFACTS / f"denoiser_{name.lower()}_seed{SEED}.pt",
        extra={"corruption": families[name].describe(), "seed": SEED, "steps": STEPS},
    )
    rows.append({"family": name, "final_train_loss": r["final_train_loss"],
                "val_loss": r["val_loss"], "identity_gap": r["identity_gap"]})

comparison = pd.DataFrame(rows).sort_values("val_loss")
io.run_or_load("denoiser_first_pass_gpt2",
                {"steps": STEPS, "batch_size": BATCH_SIZE, "lr": LR, "seed": SEED,
                 "families": list(families), "split_fingerprint": split.fingerprint(),
                 "version": 1},
                lambda: comparison)
print(comparison)

computed denoiser_first_pass_gpt2.csv  (cd5962142acba076)
  family  final_train_loss  val_loss  identity_gap
2     D3          0.021528  0.019951      3.289568
3     D4          0.215940  0.198805      6.820354
0     D1          0.197870  0.202199      6.917519
1     D2          0.288663  0.292686      6.678988


Мы видим, что при маленьком наборе D3 просто запоминает шумы, такому результату доверять нельзя

In [6]:
# Fair, apples-to-apples generalization check: same corruption for all four models, along
# directions none of them trained on. Narrowed to r in [0.05, 1.0] to match the validated
# deployment range from Step 1/3, not the full [0.05, 3] training range (r>1 is unusable).
dev_probe = corruptions.StructuredRank1(decoder, pool_indices=split.dev, holdout=set(),
                                        rho_min=0.05, rho_max=1.0)

print(f"{'family':6s} {'own val_loss':>14s}  {'dev-direction generalization':>30s}")
gen_rows = []
for name, r in results.items():
    gen_loss = denoiser.evaluate_denoiser(r["model"], val_activations, dev_probe,
                                          n_examples=4096, seed=999, device=DEVICE)
    gen_rows.append({"family": name, "own_val_loss": r["val_loss"], "dev_generalization": gen_loss})
    print(f"{name:6s} {r['val_loss']:14.4f}  {gen_loss:30.4f}")

import pandas as pd
io.run_or_load("denoiser_generalization_check_gpt2",
                {"seed": 999, "rho_min": 0.05, "rho_max": 1.0, "n_examples": 4096,
                 "split_fingerprint": split.fingerprint(), "version": 1},
                lambda: pd.DataFrame(gen_rows))

family   own val_loss    dev-direction generalization
D1             0.2022                          0.1269
D2             0.2927                          0.2246
D3             0.0200                          0.1575
D4             0.1988                          0.1207
computed denoiser_generalization_check_gpt2.csv  (51b2a81598135233)


,family,own_val_loss,dev_generalization
0,D1,0.202199,0.126894
1,D2,0.292686,0.224564
2,D3,0.019951,0.157501
3,D4,0.198805,0.120651


Берем D1 и D4 в качестве наших моделей, проводим дальнейшее тестирование

In [8]:
seed0_gen = {row["family"]: row["dev_generalization"] for row in gen_rows}

SEED2 = 1
confirmation_rows = [{"family": name, "seed": SEED, "dev_generalization": seed0_gen[name]}
                     for name in ("D1", "D4")]

for name in ("D1", "D4"):
    corruption = families[name]
    model, history = denoiser.train_denoiser(
        train_activations, corruption, d_model=768, activation_scale=scale, center=center,
        steps=STEPS, batch_size=BATCH_SIZE, lr=LR, seed=SEED2, device=DEVICE, log_every=STEPS // 10,
    )
    gen_loss = denoiser.evaluate_denoiser(model, val_activations, dev_probe,
                                          n_examples=4096, seed=999, device=DEVICE)
    confirmation_rows.append({"family": name, "seed": SEED2, "dev_generalization": gen_loss})
    denoiser.save_denoiser(model, io.ARTIFACTS / f"denoiser_{name.lower()}_seed{SEED2}.pt",
                           extra={"corruption": corruption.describe(), "seed": SEED2, "steps": STEPS})
    print(f"{name} (seed={SEED2}): dev_generalization={gen_loss:.4f}")

confirmation_df = pd.DataFrame(confirmation_rows).pivot(index="family", columns="seed",
                                                        values="dev_generalization")
confirmation_df["mean"] = confirmation_df.mean(axis=1)
print(confirmation_df.sort_values("mean"))

io.run_or_load("denoiser_second_seed_confirmation_gpt2",
                {"families": ["D1", "D4"], "seeds": [SEED, SEED2], "steps": STEPS,
                 "split_fingerprint": split.fingerprint(), "version": 1},
                lambda: confirmation_df.reset_index())

D1 (seed=1): dev_generalization=0.1252
D4 (seed=1): dev_generalization=0.1280
seed           0         1      mean
family                              
D4      0.120651  0.128018  0.124334
D1      0.126894  0.125239  0.126066
computed denoiser_second_seed_confirmation_gpt2.csv  (cb46fdaa9c579d64)


seed,family,0,1,mean
0,D1,0.126894,0.125239,0.126066
1,D4,0.120651,0.128018,0.124334


Выбираем D4 как более обоснованный вариант

## Шаг 6. Проверка зависимости качества от размера выборки

In [9]:
POOL_SIZES = [64, 256, 1024, None]  # None = full pool (this is exactly D4)
pool_rows = []

for pool_size in POOL_SIZES:
    label = str(pool_size) if pool_size else "full"
    if pool_size is None:
        corruption = corruptions.FullPoolRank1(decoder, split)
    else:
        corruption = corruptions.FixedPoolRank1(decoder, split, pool_size=pool_size, seed=0)

    model, history = denoiser.train_denoiser(
        train_activations, corruption, d_model=768, activation_scale=scale, center=center,
        steps=STEPS, batch_size=BATCH_SIZE, lr=LR, seed=0, device=DEVICE, log_every=STEPS // 10,
    )
    gen_loss = denoiser.evaluate_denoiser(model, val_activations, dev_probe,
                                          n_examples=4096, seed=999, device=DEVICE)
    pool_rows.append({"pool_size": label, "n_directions": len(corruption.pool_indices),
                      "dev_generalization": gen_loss})
    print(f"pool={label:>5s} (n={len(corruption.pool_indices):>6d}): "
          f"dev_generalization={gen_loss:.4f}")

pool_df = pd.DataFrame(pool_rows)
io.run_or_load("denoiser_pool_size_check_gpt2",
                {"pool_sizes": [str(p) for p in POOL_SIZES], "steps": STEPS, "seed": 0,
                 "split_fingerprint": split.fingerprint(), "version": 1},
                lambda: pool_df)
print(pool_df)

pool=   64 (n=    64): dev_generalization=0.1683
pool=  256 (n=   256): dev_generalization=0.1575
pool= 1024 (n=  1024): dev_generalization=0.1187
pool= full (n=130967): dev_generalization=0.1207
computed denoiser_pool_size_check_gpt2.csv  (eb7c99d6e97e1d13)
  pool_size  n_directions  dev_generalization
0        64            64            0.168250
1       256           256            0.157501
2      1024          1024            0.118651
3      full        130967            0.120651


## Шаг 7. Фиксируем параметры сравнения.

In [10]:
import random as _random

test_ids = sorted(split.test)
rng = torch.Generator().manual_seed(0)
order = torch.randperm(len(test_ids), generator=rng).tolist()
judge_concepts = sorted(test_ids[i] for i in order[:40])

judge_subset = io.run_or_load(
    "judge_subset_gpt2",
    {"n_test": len(test_ids), "n_selected": 40, "seed": 0,
     "split_fingerprint": split.fingerprint(), "version": 1},
    lambda: {"concepts": judge_concepts, "r_values": [0.2, 0.4, 0.6, 0.8]},
)
print(f"{len(judge_subset['concepts'])} concepts drawn, r={judge_subset['r_values']}")

computed judge_subset_gpt2.json  (acde5e375f7dd7c0)
40 concepts drawn, r=[0.2, 0.4, 0.6, 0.8]


In [11]:
frozen_config = {
    "corruption": "D4", "pool": "full_train_pool",
    "architecture": {"d_model": 768, "hidden_mult": 2, "t_embed_dim": 128, "center": True},
    "training": {"steps": STEPS, "batch_size": BATCH_SIZE, "lr": LR,
                "rho_min": 0.05, "rho_max": 3.0, "winner_seed": 0},
    "hook": {"model": "gpt2", "layer": LAYER},
    "r_grid": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "positions": "all_except_sink",
    "metrics": ["reference_nll", "dist_1", "dist_2", "dist_3", "repetition_4",
               "sae_concept_score", "judge_coherence", "judge_concept"],
    "judge_subset_hash": io.config_hash({"n_test": len(test_ids), "n_selected": 40, "seed": 0,
                                         "split_fingerprint": split.fingerprint(), "version": 1}),
    "split_fingerprint": split.fingerprint(),
}

frozen = io.run_or_load("frozen_method_config_gpt2",
                        {**frozen_config, "version": 1}, lambda: frozen_config)
print(f"frozen config hash: {io.config_hash({**frozen_config, 'version': 1})}")

computed frozen_method_config_gpt2.json  (49db404c4e308bc0)
frozen config hash: 49db404c4e308bc0
